In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv('../data/transactions.csv', parse_dates=['date'])
print(f"Loaded {len(df)} transactions ✅")
df.head()

Loaded 1079 transactions ✅


,date,category,amount,is_anomaly
0,2024-01-01,Utilities,1088.81,0
1,2024-01-01,Shopping,663.78,0
2,2024-01-01,Shopping,197.81,0
3,2024-01-01,Shopping,501.44,0
4,2024-01-02,Entertainment,241.91,0


In [2]:
conn = sqlite3.connect('../finance_anomaly.db')

conn.execute('''
    CREATE TABLE IF NOT EXISTS transactions (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        date        TEXT,
        category    TEXT,
        amount      REAL,
        is_anomaly  INTEGER
    )
''')

conn.execute('''
    CREATE TABLE IF NOT EXISTS monthly_summary (
        month       TEXT,
        category    TEXT,
        total_spent REAL,
        avg_spent   REAL,
        txn_count   INTEGER
    )
''')

conn.commit()
print("Database & tables created ✅")

Database & tables created ✅


In [3]:
df.to_sql('transactions', conn, if_exists='replace', index=False)
conn.commit()
print("Data loaded into SQL ✅")

Data loaded into SQL ✅


In [4]:
q1 = pd.read_sql_query('''
    SELECT strftime('%Y-%m', date) AS month,
           ROUND(SUM(amount), 2)   AS total_spent
    FROM transactions
    GROUP BY month
    ORDER BY month
''', conn)

print("Monthly Total Spend:")
print(q1.to_string(index=False))

Monthly Total Spend:
  month  total_spent
2024-01     61628.65
2024-02     54579.42
2024-03     44996.80
2024-04     34948.71
2024-05     46585.79
2024-06     48196.48
2024-07     39242.61
2024-08     49463.56
2024-09     51937.08
2024-10     38219.87
2024-11     52807.71
2024-12     42046.01


In [5]:
q2 = pd.read_sql_query('''
    SELECT category,
           ROUND(AVG(amount), 2) AS avg_amount,
           ROUND(MAX(amount), 2) AS max_amount,
           COUNT(*)              AS txn_count
    FROM transactions
    GROUP BY category
    ORDER BY avg_amount DESC
''', conn)

print("Spend by Category:")
print(q2.to_string(index=False))

Spend by Category:
     category  avg_amount  max_amount  txn_count
    Utilities     1331.36    10302.95        174
     Shopping      654.99     3807.84        195
         Food      414.51     2176.16        187
Entertainment      353.94     3526.59        166
    Transport      221.78     1621.71        170
   Healthcare      167.40     1425.08        187


In [6]:
q3 = pd.read_sql_query('''
    SELECT date, category, ROUND(amount, 2) AS amount
    FROM transactions
    ORDER BY amount DESC
    LIMIT 10
''', conn)

print("Top 10 Highest Transactions:")
print(q3.to_string(index=False))

Top 10 Highest Transactions:
               date      category   amount
2024-02-05 00:00:00     Utilities 10302.95
2024-11-28 00:00:00     Utilities  8820.30
2024-01-31 00:00:00     Utilities  7155.78
2024-08-05 00:00:00      Shopping  3807.84
2024-09-26 00:00:00      Shopping  3657.42
2024-01-19 00:00:00      Shopping  3575.52
2024-09-22 00:00:00 Entertainment  3526.59
2024-01-07 00:00:00      Shopping  2794.88
2024-08-05 00:00:00          Food  2176.16
2024-01-13 00:00:00 Entertainment  1894.08


In [7]:
q4 = pd.read_sql_query('''
    SELECT date, ROUND(SUM(amount), 2) AS daily_total
    FROM transactions
    GROUP BY date
    HAVING daily_total > (
        SELECT AVG(daily_avg) * 3
        FROM (
            SELECT date, SUM(amount) AS daily_avg
            FROM transactions
            GROUP BY date
        )
    )
    ORDER BY daily_total DESC
''', conn)

print(f"Days with suspiciously high spend: {len(q4)}")
print(q4.to_string(index=False))

Days with suspiciously high spend: 5
               date  daily_total
2024-02-05 00:00:00     11660.35
2024-11-28 00:00:00      9468.43
2024-01-31 00:00:00      7919.01
2024-08-05 00:00:00      7068.44
2024-09-26 00:00:00      4786.76


In [8]:
q5 = pd.read_sql_query('''
    SELECT strftime('%Y-%m', date) AS month,
           SUM(is_anomaly)         AS known_anomalies
    FROM transactions
    GROUP BY month
    ORDER BY month
''', conn)

print("Monthly Anomaly Distribution:")
print(q5.to_string(index=False))

conn.close()
print("\nDatabase connection closed ✅")

Monthly Anomaly Distribution:
  month  known_anomalies
2024-01                5
2024-02                3
2024-03                2
2024-04                1
2024-05                1
2024-06                1
2024-07                0
2024-08                3
2024-09                2
2024-10                0
2024-11                2
2024-12                0

Database connection closed ✅
